# NETOPT-EAST: Phase 2 — Chance-Constrained MILP Optimization
**Notebook:** `notebooks/03_model_formulation.ipynb`  
**Core Innovation:**  
> *Safety stock is integrated directly into the MILP optimization model via Sample Average Approximation (SAA) chance constraints and correlated pooling, rather than calculated post-hoc.*

---

## 1. Mathematical Formulation & Code Mapping

### Sets & Indices
- $I = \{1, \dots, N\}$: Set of 40 demand nodes (Eastern India)
- $J = \{1, \dots, M\}$: Set of 6 candidate warehouses (Kolkata, Bhubaneswar, Patna, Ranchi, Guwahati, Siliguri)
- $S = \{1, \dots, 50\}$: Set of correlated demand shock scenarios

### Decision Variables
- $y_j \in \{0, 1\}$: Binary variable indicating if warehouse $j$ is open
- $x_{ji} \ge 0$: Weekly unit flow shipped from warehouse $j$ to demand node $i$
- $q_j \ge 0$: Weekly inbound throughput received at warehouse $j$ from the plant
- $I_j \ge 0$: Lead-time inventory units held at warehouse $j$

### Objective Function
$$\min \sum_{j \in J} f_j y_j + 52 \sum_{j \in J} \sum_{i \in I} c_{ji} x_{ji} + 52 \sum_{j \in J} c_{pj} q_j + 52 \sum_{j \in J} v_j q_j + h \sum_{j \in J} W_{\text{cycle}} q_j + h z_\alpha \sum_{j \in J} \sigma_{\text{pool}}(j) \sqrt{L_j} y_j$$

Where:
- $f_j$: Annual fully loaded fixed facility cost (`warehouse_rents.csv`)
- $c_{ji}$: Outbound secondary LTL freight per unit (`transport_costs.csv`)
- $c_{pj}$: Inbound primary FTL freight per unit from the plant
- $v_j$: Variable handling cost per unit (₹1.0/unit piece-rate labor)
- $h$: Annual holding cost rate ($h = 0.25 \times \text{SKU Value}$)
- $W_{\text{cycle}}$: Cycle inventory coverage weeks ($3.0$ weeks)
- $\sigma_{\text{pool}}(j) = \sqrt{\boldsymbol{\sigma}_j^T \boldsymbol{\Sigma}_j \boldsymbol{\sigma}_j}$: Correlated pooled standard deviation of nodes assigned to warehouse $j$

### Constraints
1. **Demand Satisfaction:** $\sum_{j \in J} x_{ji} \ge d_i \quad \forall i \in I$
2. **Flow Conservation:** $q_j = \sum_{i \in I} x_{ji} \quad \forall j \in J$
3. **Throughput Capacity:** $q_j \le C_j y_j \quad \forall j \in J$
4. **Big-M Flow Linking:** $x_{ji} \le d_i y_j \quad \forall j \in J, i \in I$
5. **SAA Chance Constraint:** $I_j \ge \sum_{i \in I} x_{ji} \varepsilon_i^s L_j - M (1 - y_j) \quad \forall j \in J, s \in S$
6. **Closed Inventory Elimination:** $I_j \le 4 C_j y_j \quad \forall j \in J$


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from ortools.sat.python import cp_model

from src.optimization.model import solve_network_design
from src.optimization.inventory import pooled_std_for_assignment
from src.optimization.scenarios import generate_scenarios

print("Environment and optimization modules loaded successfully.")


In [ ]:
# Load Real-Anchored Network Data
nodes_df = pd.read_csv('../data/processed/demand_nodes_anchored.csv')
whs_df = pd.read_csv('../data/processed/candidate_warehouses.csv')
lanes_df = pd.read_csv('../data/processed/transport_costs.csv')
corr_df = pd.read_csv('../data/processed/correlation_matrix.csv', index_col=0)

print(f"Loaded {len(nodes_df)} demand nodes, {len(whs_df)} candidate facilities, and {len(lanes_df)} transport lanes.")
display(whs_df[['warehouse_id', 'city', 'fixed_cost_monthly_rupees', 'capacity_units_per_week', 'variable_cost_per_unit', 'lead_time_from_plant_days']])


In [ ]:
# Solve Chance-Constrained MILP at 95% Service Level
solution = solve_network_design(
    nodes=nodes_df,
    warehouses=whs_df,
    transport_costs=lanes_df,
    corr_matrix=corr_df,
    service_level=0.95,
    holding_cost_rate=0.25,
    product_value=90.0,
    n_scenarios=50,
    verbose=True
)


In [ ]:
import json
with open('../data/processed/baseline_cost.json', 'r') as f:
    baseline = json.load(f)

b_tot = baseline['total_annual_cost']
opt_tot = solution['total_annual_cost']
savings = b_tot - opt_tot
pct_savings = (savings / b_tot) * 100

print(f"\n=================== EXECUTIVE COST COMPARISON ===================")
print(f"Baseline Network Cost (Direct Plant LTL):  ₹{b_tot:,.0f}")
print(f"Optimized Network Cost (95% SL Hubs):     ₹{opt_tot:,.0f}")
print(f"Net Annual Landed Cost Savings:           ₹{savings:,.0f} ({pct_savings:.2f}% reduction)")
print(f"================================================================")

# Cost Breakdown Donut Chart
cost_categories = ['Outbound Transport', 'Inbound FTL Freight', 'Fixed Facility Costs', 'Variable Handling', 'Inventory Holding']
cost_values = [
    solution['transport_cost'],
    solution['inbound_cost'],
    solution['fixed_cost'],
    solution['variable_cost'],
    solution['inventory_cost']
]

fig_cost = go.Figure(data=[go.Pie(
    labels=cost_categories,
    values=cost_values,
    hole=.45,
    textinfo='label+percent',
    marker_colors=['#2b5c8f', '#4682b4', '#e67e22', '#f39c12', '#27ae60']
)])
fig_cost.update_layout(title="Optimal Network Cost Breakdown (95% Service Level)", width=700, height=500)
fig_cost.show()


In [ ]:
# Detailed Warehouse Throughput and Inventory Buffer Summary
inv_table = solution['inventory'].copy()
flows_df = solution['flows']
throughput_by_wh = flows_df.groupby('warehouse')['weekly_units'].sum().reset_index()

inv_table = inv_table.merge(throughput_by_wh, on='warehouse')
inv_table['utilization_pct'] = (inv_table['weekly_units'] / whs_df.set_index('warehouse_id').loc[inv_table['warehouse'], 'capacity_units_per_week'].values) * 100

print("=== FACILITY UTILIZATION & INVENTORY DETAILS ===")
display(inv_table[['warehouse', 'weekly_units', 'utilization_pct', 'cycle_stock', 'safety_stock', 'total_inventory', 'annual_holding_cost']])


In [ ]:
# Automated Execution & Reporting of Hand-Solved Test Cases TC-001..3
import pytest
from tests.regression.test_benchmarks import (
    test_tc_001_location_transport_tradeoff,
    test_tc_002_pooling_verification,
    test_tc_003_single_warehouse_single_node
)

print("Running TC-003 (Single WH, Single Node End-to-End)...")
test_tc_003_single_warehouse_single_node()
print("  ✓ TC-003 PASSED: Exact match with analytical hand calculations.")

print("\nRunning TC-001 (2 WH, 3 Nodes Location-Transport Tradeoff)...")
test_tc_001_location_transport_tradeoff()
print("  ✓ TC-001 PASSED: Model correctly selects WH2 only, optimizing fixed vs transport cost.")

print("\nRunning TC-002 (Inventory Pooling & Spatial Correlation)...")
test_tc_002_pooling_verification()
print("  ✓ TC-002 PASSED: Analytical pooling benefits correctly match rho=0 and rho=1 limits.")

print("\nALL PHASE 2 VALIDATION TEST CASES VERIFIED SUCCESSFULLY!")
